In [5]:
import pandas as pd

df = pd.read_csv("../data/train_custom.csv")

print(len(df))
df.head()

13158


,tweet_id,sentiment,text
0,39361,positive,Fake news Muslim man was not beaten up in Kari...
1,19737,negative,@ VazeIndian @ RahulGandhi Muslim vo kutte ki ...
2,12664,negative,@ margallass Aray nhe Nawaz sharif sala dulla ...
3,27674,negative,RT @ shaik _ hussam EVM ki rigging nahi hui Hi...
4,18543,negative,@ bprerna Prerna bakshi tum jsise log ho .. ji...


In [ ]:
import openai
import time
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

client = openai.OpenAI(api_key="KEY_API")

EMOTION_PROMPT = """You are an emotion classifier for code-mixed Hinglish social media text.
Classify the following tweet into exactly one of these emotions: anger, joy, sadness, trust, fear, surprise.
Respond with only the emotion word, nothing else.

Tweet: {tweet}
Emotion:"""

def label_emotion(row):
    tweet_id, text = row
    for attempt in range(3):  # 3 retries on rate limit
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": EMOTION_PROMPT.format(tweet=text)}],
                temperature=0,
                max_tokens=10
            )
            label = response.choices[0].message.content.strip().lower()
            return tweet_id, label
        except openai.RateLimitError:
            time.sleep(2 ** attempt)  # 1s, 2s, 4s backoff
        except Exception as e:
            return tweet_id, "error"
    return tweet_id, "error"

In [10]:
SAVE_PATH = "../data/emotion_train_labeled.csv"

if os.path.exists(SAVE_PATH):
    already_done = pd.read_csv(SAVE_PATH)
    done_ids = set(already_done['tweet_id'].tolist())
    print(f"Resuming — {len(done_ids)} already labeled")
else:
    already_done = pd.DataFrame()
    done_ids = set()

remaining = df[~df['tweet_id'].isin(done_ids)][['tweet_id', 'text']]
print(f"Tweets remaining: {len(remaining)}")

rows = list(remaining.itertuples(index=False, name=None))
results = []
BATCH_SIZE = 200  # 200 per batch

for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i:i+BATCH_SIZE]
    
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(label_emotion, row): row for row in batch}
        for future in as_completed(futures):
            tweet_id, label = future.result()
            results.append({'tweet_id': tweet_id, 'label': label})
    
    # Save after every batch
    partial = pd.DataFrame(results)
    combined = pd.concat([already_done, partial], ignore_index=True)
    combined.to_csv(SAVE_PATH, index=False)
    
    print(f"Done: {min(i + BATCH_SIZE, len(rows))} / {len(rows)}")
    time.sleep(3)  # 3 second gap between batches

print("✓ Labeling complete!")

Resuming — 4400 already labeled
Tweets remaining: 8758


KeyboardInterrupt: 

In [8]:
done = pd.read_csv("../data/emotion_train_labeled.csv")
print(f"Saved so far: {len(done)}")

Saved so far: 4400


In [11]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say the word: joy"}],
    temperature=0,
    max_tokens=10
)
print(response.choices[0].message.content)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hiV86l554TdHLwwdzNFr2Pnt on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}

In [15]:
import pandas as pd

df1 = pd.read_csv("../data/emotion_labeled_2.csv")
df2 = pd.read_csv("../data/emotion_labeled_1.csv")

print("df1 columns:", df1.columns.tolist())
print("df2 columns:", df2.columns.tolist())
print("\ndf1 shape:", df1.shape)
print("df2 shape:", df2.shape)

df1 columns: ['tweet_id', 'label']
df2 columns: ['tweet_id', 'sentiment', 'text', 'emotion']

df1 shape: (4400, 2)
df2 shape: (2825, 4)


In [16]:
# Get text back for df1
train_original = pd.read_csv("../data/train_custom.csv")
df1 = df1.merge(train_original[['tweet_id', 'text']], on='tweet_id', how='left')
df1 = df1.rename(columns={'label': 'emotion'})

# Keep only what we need from df2
df2 = df2[['tweet_id', 'text', 'emotion']]

# Merge
merged = pd.concat([df1, df2], ignore_index=True)
print(merged.columns.tolist())
print(merged.shape)
print(merged['emotion'].value_counts())

['tweet_id', 'emotion', 'text']
(7225, 3)
emotion
anger        3182
joy          2635
sadness       560
trust         392
error         324
surprise      113
fear           18
confusion       1
Name: count, dtype: int64


In [17]:
from sklearn.model_selection import train_test_split

# Clean
keep = ['anger', 'joy', 'sadness', 'trust']
merged_clean = merged[merged['emotion'].isin(keep)].reset_index(drop=True)
print("After cleaning:", merged_clean['emotion'].value_counts())
print("Total:", len(merged_clean))

# Split 85/15
train, dev = train_test_split(merged_clean, test_size=0.15, 
                               stratify=merged_clean['emotion'], 
                               random_state=42)

print(f"\nTrain: {len(train)}")
print(f"Dev: {len(dev)}")

# Save
train.to_csv("../data/emotion_train_final.csv", index=False)
dev.to_csv("../data/emotion_dev_final.csv", index=False)

print("✓ Saved!")

After cleaning: emotion
anger      3182
joy        2635
sadness     560
trust       392
Name: count, dtype: int64
Total: 6769

Train: 5753
Dev: 1016
✓ Saved!
